# P5 · Transfer learning: adapt a pretrained network to your own photos

**What we do today**

1. The idea: reuse a network that others have trained on more than a million photos.
2. The data: photos of 5 breeds of cats and dogs, only 60 per breed.
3. A baseline: a small network trained from nothing on the same photos (it does poorly).
4. The recipe, in five steps: download a pretrained network, freeze it, give it a new classifier, train the
   classifier, then unfreeze its last block and train a little more.
5. Use the adapted network: name the breed in a photo file, save the network to a file and load it again.
6. Change the settings yourself and run the whole recipe again (section 10, **Try it**).

**How to use this notebook.** Click into the first code cell and press **Shift+Enter** to run it; that moves
you to the next cell. Run the cells from top to bottom: later cells use what earlier cells made. You type
three short lines of code today, in steps 2, 3 and 5. Each is marked `____` (four underscores) with a comment
`# YOUR CODE:` that says exactly what to write. The cell after it checks your line and prints `[PASS]` or
`[FAIL]` with a hint. If you leave a blank as it is, the check does the step for you and says so ("done for
you so you can continue"), so the notebook always runs to the end. Cells marked **Try it** hold settings you
may change. The whole notebook runs in about 2 to 5 minutes on a laptop; every training cell says how long it
takes.

* `P.something(...)` is a function of the course toolkit (P1 introduced it): it does the plumbing (drawing,
  timing, the training loop) so that you can concentrate on the recipe. Each one is explained in a sentence
  where it is first used.
* `T.something(...)` is one of today's checks.
* Closed boxes **Functions used here (click to open)** explain the PyTorch functions in the next cell, with
  a small example. Open them when you want the details.
* **New words** are in bold where they are explained. All of them are collected at the end.
* Numbers quoted in the text are what the cells print. On your computer the last digit can differ a little,
  and times depend on how fast your computer is and on what else it is doing.

## 0 · Setup

The first cell prepares everything, and must run first every time you open the notebook:

* the `while` loop finds the folder `cvcourse`, the course's helper code, and lets Python import it;
* `torch` is **PyTorch**, `torch.nn` (short `nn`) its building blocks (the layers), and `torchvision` the
  PyTorch library for images: it holds the pretrained networks we download;
* `P` is the course toolkit and `T` today's checks;
* `P.setup(seed=0)` fixes the **random seed**, so that every run gives the same results, and prints whether
  PyTorch computes on the CPU or on a graphics card (a CPU is fine);
* `____ = None` creates the blank you will replace three times. Until you replace it, a blank does nothing.

In [ ]:
import sys
import pathlib

folder = pathlib.Path.cwd()
while not (folder / "cvcourse").is_dir() and folder != folder.parent:
    folder = folder.parent                      # go up until the folder that contains cvcourse
sys.path.insert(0, str(folder))

import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
from PIL import Image
from cvcourse import torchkit
from cvcourse import practical as P
from cvcourse import ptests_p5 as T

P.setup(seed=0)
____ = None                                     # the blank: replace it where a cell says YOUR CODE

## 1 · The idea

Training a good image network from nothing needs a lot of photos. **ResNet-18**, the network we use today,
learned from **ImageNet**: 1.28 million photos sorted into 1 000 classes (animals, vehicles, food, tools,
...), for days on graphics cards. What it learned is not only about those 1 000 classes. Its first layers
respond to edges and colours, the next ones to textures (fur, grass, metal), the deepest ones to patterns
and object parts (eyes, ears, noses, wheels). That knowledge is useful for almost any photo.

**Transfer learning** reuses it. A classification network has two parts:

* the **feature extractor**: all the layers except the last. It turns a photo into a list of numbers that
  describe it, its **features** (for ResNet-18: 512 numbers);
* the **classifier** (also called the **head**): the last layer. It turns the 512 features into one score
  per class (for ImageNet: 1 000 scores).

Keep the feature extractor as it is and replace only the classifier by a new one for *your* classes. The new
layer is the only part that has to learn, and it learns from a few hundred photos in about a minute.

In [ ]:
T.show_idea()

The light blue blocks are **frozen**: copied as they are, training does not change them. Only the new green
classifier learns. The dashed block, `layer4`, is the optional last step: once the classifier works, let the
last block of the feature extractor adapt a little too.

## 2 · The data: 5 breeds, 60 photos each

**Oxford-IIIT Pet** is a collection of 7 390 photos of 37 breeds of cats and dogs, about 200 per breed, taken
by their owners: all poses, backgrounds and lighting. The photos are already in the folder
`data/oxford-iiit-pet/images`. Every file name starts with the breed, for example `pug_12.jpg` or
`Siamese_7.jpg` (cat breeds start with a capital letter, dog breeds do not).

We take 5 breeds that look different and only **60 training photos per breed, 300 in total**. That is very
little: ImageNet has about 1 300 photos per class, and P1 trained on 10 000 digits. The **validation photos**
are 40 other photos of each breed (200 in total): never used for training, only to measure how well the
network does on photos it has not seen. The breeds are the **classes**: the possible answers.

The loader below is plain code that you can copy into your own project:

* `preparation(image_size)` lists the steps that turn a photo into the numbers a network gets: scale it and
  cut out a square of `image_size` x `image_size` pixels, turn it into a **tensor** (PyTorch's array of
  numbers) of shape 3 x height x width with values 0 to 1, then **normalise** each colour channel: subtract
  the ImageNet average and divide by the ImageNet spread (step 1 explains why exactly these numbers).
  `v2.Compose([...])` chains the steps in order.
* `load_pets(...)` finds the files of each breed, takes the first ones for training and the last ones for
  validation, reads and prepares every photo, and returns two loaders. It takes the breeds in turn (photo 1 of
  every breed, then photo 2, ...), so that the breeds are mixed in the pictures below.
* `P.ImageSet(images, labels, classes, mean, std)` keeps the prepared photos and their **labels** (the breed
  number of each photo: 0 for the first breed, 1 for the second, ...) in memory; `P.loaders(...)` hands them
  out in **batches** of 32. The training loader shuffles them in every **epoch** (one pass over all training
  photos).

<details class="api-ref" data-api="torchkit.data_root,PIL.Image.open,torchvision.transforms.v2.Compose,torchvision.transforms.v2.Resize,torchvision.transforms.v2.CenterCrop,torchvision.transforms.v2.ToImage,torchvision.transforms.v2.ToDtype,torchvision.transforms.v2.Normalize,torch.stack,torch.tensor"><summary><b>Functions used here</b> (click to open): <code>torchkit.data_root</code> · <code>PIL.Image.open</code> · <code>torchvision.transforms.v2.Compose</code> · <code>torchvision.transforms.v2.Resize</code> · <code>torchvision.transforms.v2.CenterCrop</code> · <code>torchvision.transforms.v2.ToImage</code> · <code>torchvision.transforms.v2.ToDtype</code> · <code>torchvision.transforms.v2.Normalize</code> · <code>torch.stack</code> · <code>torch.tensor</code></summary>
<p><b><code>torchkit.data_root()</code></b><br>Course helper. Returns the folder where all datasets and checkpoints live, as a pathlib.Path: data/ at the repository root (or the folder named in the CVCOURSE_DATA environment variable, if set), created if it is missing. torchkit.CHECKPOINTS is data_root() / &quot;checkpoints&quot;.</p><p><b><code>photo = Image.open(path).convert(&quot;RGB&quot;)</code></b><br>Opens an image file (JPEG, PNG, ...) as a PIL image; `from PIL import Image` first. PIL (the &quot;Pillow&quot; library) is what torchvision uses to read photos. .convert(&quot;RGB&quot;) makes sure the photo has exactly 3 colour channels: a few photos are stored in grey (1 channel) or with a transparency channel (4), and a network trained on colour photos needs 3. photo.size is (width, height).</p><pre>&gt;&gt;&gt; from PIL import Image
&gt;&gt;&gt; import io
&gt;&gt;&gt; file = io.BytesIO()                                # an in-memory file; in practice a path &quot;pug_1.jpg&quot;
&gt;&gt;&gt; Image.new(&quot;L&quot;, (40, 30)).save(file, format=&quot;PNG&quot;)  # a grey 40 x 30 picture
&gt;&gt;&gt; photo = Image.open(file)
&gt;&gt;&gt; photo.mode, photo.size                             # 1 grey channel; (width, height)
(&#x27;L&#x27;, (40, 30))
&gt;&gt;&gt; photo.convert(&quot;RGB&quot;).mode                          # now 3 colour channels
&#x27;RGB&#x27;</pre><p><i>Watch out:</i> PIL sizes are (width, height), tensors are (channels, height, width): v2.ToImage() does the conversion.</p><p>See also: <code>torchvision.transforms.v2.ToImage</code>, <code>torchvision.datasets.ImageFolder</code></p><p><b><code>v2.Compose([transform1, transform2, ...])</code></b><br>Chains transforms into one: calling the result applies them in the listed order, each on the output of the previous one. A whole per-image pipeline (resize, augment, convert to float, normalise) becomes one callable that a Dataset applies to every sample.</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; tf = v2.Compose([v2.Resize(16), v2.ToDtype(torch.float32, scale=True)])
&gt;&gt;&gt; img = torch.randint(0, 256, (3, 32, 48), dtype=torch.uint8)
&gt;&gt;&gt; out = tf(img)
&gt;&gt;&gt; out.shape, out.dtype, 0 &lt;= float(out.min()) &lt;= float(out.max()) &lt;= 1
(torch.Size([3, 16, 24]), torch.float32, True)</pre><p><i>Watch out:</i> Order matters: Normalize needs a float tensor, so ToDtype(torch.float32, scale=True) must come before it; the cheap geometric and colour transforms go first, on uint8.</p><p>See also: <code>torchvision.transforms.v2.ToDtype</code>, <code>torchvision.transforms.v2.Normalize</code></p><p><b><code>v2.Resize(size, antialias=True)</code></b><br>Resizes an image tensor (C, H, W) or a batch (N, C, H, W). With one number the shorter side becomes that size and the aspect ratio is kept; with a pair (h, w) the output is exactly h x w. The dtype is kept: uint8 in, uint8 out.</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; img = torch.randint(0, 256, (3, 32, 48), dtype=torch.uint8)    # H 32, W 48
&gt;&gt;&gt; v2.Resize(16)(img).shape                   # shorter side -&gt; 16, ratio kept
torch.Size([3, 16, 24])
&gt;&gt;&gt; v2.Resize((16, 16))(img).shape             # exact size: the image is squashed
torch.Size([3, 16, 16])
&gt;&gt;&gt; v2.Resize(16)(img).dtype
torch.uint8</pre><p><i>Watch out:</i> Resize(128) gives a square image only if the input is square; follow it with CenterCrop(128) (or use RandomResizedCrop) when every image must have the same size.</p><p>See also: <code>torchvision.transforms.v2.CenterCrop</code>, <code>torchvision.transforms.v2.RandomResizedCrop</code>, <code>F.interpolate</code></p><p><b><code>v2.CenterCrop(size)</code></b><br>Cuts out the central size x size region (or (h, w)): no resizing, no randomness. The evaluation counterpart of RandomResizedCrop: Resize(s) followed by CenterCrop(s) gives a square image whose side is s.</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; img = torch.randint(0, 256, (3, 32, 48), dtype=torch.uint8)
&gt;&gt;&gt; v2.CenterCrop(32)(img).shape               # 8 columns cut off on each side
torch.Size([3, 32, 32])
&gt;&gt;&gt; torch.equal(v2.CenterCrop(32)(img), img[:, :, 8:40])
True
&gt;&gt;&gt; v2.Compose([v2.Resize(16), v2.CenterCrop(16)])(img).shape
torch.Size([3, 16, 16])</pre><p>See also: <code>torchvision.transforms.v2.Resize</code>, <code>torchvision.transforms.v2.RandomResizedCrop</code></p><p><b><code>v2.ToImage()</code></b><br>Transform that turns a PIL image or an (H, W, C) NumPy array into an image tensor of shape (C, H, W). Values and dtype are kept (uint8 stays 0..255), so it is almost always followed by v2.ToDtype(torch.float32, scale=True) to get floats in [0, 1].</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; from PIL import Image
&gt;&gt;&gt; img = v2.ToImage()(Image.new(&quot;RGB&quot;, (48, 32)))       # PIL sizes are (width, height)
&gt;&gt;&gt; img.shape, img.dtype                                 # C, H, W; still uint8
(torch.Size([3, 32, 48]), torch.uint8)
&gt;&gt;&gt; v2.ToImage()(np.zeros((32, 48, 3), np.uint8)).shape  # an H, W, C array
torch.Size([3, 32, 48])</pre><p>See also: <code>torchvision.transforms.v2.ToDtype</code>, <code>torchvision.transforms.v2.Compose</code></p><p><b><code>v2.ToDtype(torch.float32, scale=True)</code></b><br>Converts the image to another dtype. With scale=True it also rescales the values to the range of the new type: uint8 0 .. 255 becomes float 0.0 .. 1.0. The step between the cheap uint8 transforms and Normalize.</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; img = torch.tensor([[[0, 128, 255]]], dtype=torch.uint8)
&gt;&gt;&gt; v2.ToDtype(torch.float32, scale=True)(img)
tensor([[[0.0000, 0.5020, 1.0000]]])
&gt;&gt;&gt; v2.ToDtype(torch.float32)(img)             # scale=False: same numbers, now float
tensor([[[  0., 128., 255.]]])</pre><p><i>Watch out:</i> Without scale=True the values stay 0 .. 255, and Normalize with the ImageNet mean / std (made for 0 .. 1) then gives values around 1000: the network sees nonsense, and nothing raises an error.</p><p>See also: <code>torchvision.transforms.v2.Normalize</code>, <code>Tensor.float</code></p><p><b><code>v2.Normalize(mean, std)</code></b><br>Standardises each channel c: out[c] = (x[c] - mean[c]) / std[c]. With the ImageNet statistics (torchkit.IMAGENET_MEAN, torchkit.IMAGENET_STD) the pixel values go from [0, 1] to about [-2.1, 2.6], centred near 0, which is what ImageNet-trained networks expect. Needs a float tensor.</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; norm = v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
&gt;&gt;&gt; norm(torch.ones(3, 1, 1)).flatten()                     # a white pixel
tensor([2.2489, 2.4286, 2.6400])
&gt;&gt;&gt; norm(torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)).flatten()    # the mean colour -&gt; 0
tensor([0., 0., 0.])</pre><p><i>Watch out:</i> On a uint8 image it raises TypeError (&quot;Input tensor should be a float tensor&quot;): put ToDtype(torch.float32, scale=True) before it. To display a normalised image, undo it with torchkit.denormalize.</p><p>See also: <code>torchvision.transforms.v2.ToDtype</code>, <code>torchkit.denormalize</code></p><p><b><code>torch.stack(tensors, dim=0)</code></b><br>Joins a list of tensors of identical shape along a new dimension: N tensors of shape (C, H, W) give one tensor (N, C, H, W). This is what a DataLoader does with the items of a batch.</p><pre>&gt;&gt;&gt; imgs = [torch.zeros(3, 4, 4), torch.ones(3, 4, 4)]
&gt;&gt;&gt; torch.stack(imgs).shape
torch.Size([2, 3, 4, 4])
&gt;&gt;&gt; torch.stack(imgs, dim=1).shape         # the new axis can go anywhere
torch.Size([3, 2, 4, 4])
&gt;&gt;&gt; torch.stack([torch.zeros(3, 4, 6), torch.zeros(3, 5, 5)])
Traceback (most recent call last):
...
RuntimeError: stack expects each tensor to be equal size, but got [3, 4, 6] at entry 0 and [3, 5, 5] at entry 1</pre><p><i>Watch out:</i> torch.cat joins along an existing dimension instead: two (2, 3) tensors give (4, 3) with cat and (2, 2, 3) with stack.</p><p>See also: <code>torch.cat</code>, <code>torchkit.detection_collate</code></p><p><b><code>torch.tensor(data, dtype=None, device=None)</code></b><br>Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</p><pre>&gt;&gt;&gt; torch.tensor([1, 2, 3]).dtype, torch.tensor([1., 2., 3.]).dtype
(torch.int64, torch.float32)
&gt;&gt;&gt; torch.tensor(2.0)                      # one number: a 0-dimensional tensor, shape ()
tensor(2.)
&gt;&gt;&gt; torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1).shape   # per-channel values, ready to broadcast
torch.Size([3, 1, 1])</pre><p><i>Watch out:</i> An integer tensor cannot be averaged: torch.tensor([1, 2]).mean() raises an error. Write the numbers as floats (1.) or pass dtype=torch.float32.</p><p>See also: <code>torch.from_numpy</code>, <code>Tensor.view</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]       # the average red, green and blue of the ImageNet photos (0 to 1)
IMAGENET_STD = [0.229, 0.224, 0.225]        # and their spread (standard deviation)


def preparation(image_size):
    """The steps that turn one photo into what an ImageNet network expects."""
    return v2.Compose([
        v2.Resize(image_size),                      # scale the photo: its shorter side becomes image_size pixels
        v2.CenterCrop(image_size),                  # cut out the central square, image_size x image_size
        v2.ToImage(),                               # photo -> tensor 3 x height x width, values 0 to 255
        v2.ToDtype(torch.float32, scale=True),      # -> decimal numbers 0 to 1
        v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),  # -> (value - mean) / std, per colour channel
    ])


def load_pets(breeds, photos_per_class, val_per_class, image_size):
    """Training and validation loaders for a few Oxford-IIIT Pet breeds."""
    folder = torchkit.data_root() / "oxford-iiit-pet" / "images"
    prepare = preparation(image_size)
    train_files = []                                # train_files[label] = the training files of that breed
    val_files = []
    for breed in breeds:
        found = sorted(folder.glob(breed + "_*.jpg"))         # every photo of this breed, e.g. pug_1.jpg ...
        if len(found) < photos_per_class + val_per_class:
            raise ValueError(f"'{breed}': {len(found)} photos found, {photos_per_class + val_per_class} needed. "
                             f"Check the spelling (section 10 lists the breeds) or ask for fewer photos.")
        train_files.append(found[:photos_per_class])          # the first photos: training
        val_files.append(found[-val_per_class:])              # the last photos: validation, never trained on

    train_images, train_labels = [], []
    for k in range(photos_per_class):               # photo k of every breed in turn, so the breeds are mixed
        for label in range(len(breeds)):
            photo = Image.open(train_files[label][k]).convert("RGB")
            train_images.append(prepare(photo))
            train_labels.append(label)
    val_images, val_labels = [], []
    for k in range(val_per_class):
        for label in range(len(breeds)):
            photo = Image.open(val_files[label][k]).convert("RGB")
            val_images.append(prepare(photo))
            val_labels.append(label)

    train_set = P.ImageSet(torch.stack(train_images), torch.tensor(train_labels), breeds,
                           IMAGENET_MEAN, IMAGENET_STD, name="Oxford-IIIT Pet")
    val_set = P.ImageSet(torch.stack(val_images), torch.tensor(val_labels), breeds,
                         IMAGENET_MEAN, IMAGENET_STD, name="Oxford-IIIT Pet")
    print(f"{len(breeds)} breeds: {len(train_set)} training photos, {len(val_set)} validation photos, "
          f"each 3 x {image_size} x {image_size}")
    return P.loaders(train_set, val_set, batch_size=32)

Now load 5 breeds at 128 x 128 pixels and look at the first 20 training photos. `number_of_classes` is the
number of breeds: we need it for the classifiers.

**Expected time: a few seconds.**

In [ ]:
breeds = ["Bombay", "Siamese", "pug", "samoyed", "beagle"]   # the classes: how their file names start
image_size = 128
train_loader, val_loader = load_pets(breeds, photos_per_class=60, val_per_class=40, image_size=image_size)
number_of_classes = len(breeds)
P.show_samples(train_loader, breeds, n=20, cols=10)

A Bombay is a black cat, a Siamese a cream cat with dark ears and face, a pug a small dog with a flat black
face, a samoyed a big white fluffy dog, a beagle a brown, black and white hound. Easy for us, but the network
has to learn it from 60 photos per breed, all different in pose and background.

<details>
<summary><b>Your own photos</b> (click to open)</summary>

Sort them into one folder per class, for example `my_photos/train/cat/*.jpg` and `my_photos/train/dog/*.jpg`,
and a second set of other photos in `my_photos/val/cat` and `my_photos/val/dog`. Then
`torchvision.datasets.ImageFolder` does the work of `load_pets` (it reads the photos as the loader asks for
them; the class names are the folder names, in alphabetical order):

```python
train_set = torchvision.datasets.ImageFolder("my_photos/train", transform=preparation(128))
val_set = torchvision.datasets.ImageFolder("my_photos/val", transform=preparation(128))
train_loader, val_loader = P.loaders(train_set, val_set, batch_size=32)
```

`train_set.classes` lists the class names; use it wherever this notebook uses `breeds`. The rest of the
recipe stays the same.
</details>

<details class="api-ref" data-api="torchvision.datasets.ImageFolder"><summary><b>Functions used here</b> (click to open): <code>torchvision.datasets.ImageFolder</code></summary>
<p><b><code>ds = torchvision.datasets.ImageFolder(root, transform=None)</code></b><br>A dataset of your own photos, sorted into one sub-folder per class: root/cat/*.jpg, root/dog/*.jpg, ... ds.classes lists the class names (the folder names, in alphabetical order); ds[i] is (image, class index), with `transform` (for example a v2.Compose of resize, ToImage, ToDtype, Normalize) applied to the image as it is read. Put training and validation photos in two separate root folders, each with the same class sub-folders, and wrap each dataset in a DataLoader.</p><pre>&gt;&gt;&gt; import pathlib, tempfile
&gt;&gt;&gt; from PIL import Image
&gt;&gt;&gt; root = pathlib.Path(tempfile.mkdtemp())
&gt;&gt;&gt; for name in [&quot;dog&quot;, &quot;cat&quot;]:
...     (root / name).mkdir()
...     Image.new(&quot;RGB&quot;, (8, 6)).save(root / name / &quot;photo1.jpg&quot;)
&gt;&gt;&gt; ds = torchvision.datasets.ImageFolder(root)
&gt;&gt;&gt; ds.classes, len(ds)                 # alphabetical: cat = class 0, dog = class 1
([&#x27;cat&#x27;, &#x27;dog&#x27;], 2)
&gt;&gt;&gt; image, label = ds[1]
&gt;&gt;&gt; image.size, label                   # a PIL image (no transform given) and its class index
((8, 6), 1)
&gt;&gt;&gt; import shutil; shutil.rmtree(root)  # tidy up the example folder</pre><p>See also: <code>torch.utils.data.DataLoader</code>, <code>torchvision.transforms.v2.Compose</code>, <code>PIL.Image.open</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

## 3 · A baseline: a small network trained from scratch

To see what transfer learning brings, first train a network **from scratch**, that is, starting from random
weights, on the same 300 photos. It is a small convolutional network like network B of P1, made for colour
photos: four convolutions (3 colour channels in, 16, 32, 64, 64 feature maps), each followed by ReLU, the
first three by max pooling (128 -> 64 -> 32 -> 16 pixels).

One new layer: `nn.AdaptiveAvgPool2d(1)` averages each of the 64 feature maps over the whole picture into one
number (**global average pooling**), so 64 numbers reach the final linear layer whatever the picture size.
ResNet-18 ends in the same way.

`P.train(model, train_loader, val_loader, epochs=..., lr=...)` trains the network (P1 explains it): one line
per epoch, and the curves of the **loss** (how wrong the scores are, lower is better) and the **accuracy**
(the percentage of photos whose highest score is the right breed) for the training photos (blue) and the
validation photos (orange). `lr` is the **learning rate**: how big each change of the weights is.

**Expected time: about 15 seconds** (up to a minute on a slow laptop).

<details class="api-ref" data-api="torch.manual_seed,nn.Sequential,nn.Conv2d,nn.ReLU,nn.MaxPool2d,nn.AdaptiveAvgPool2d,nn.Flatten,nn.Linear"><summary><b>Functions used here</b> (click to open): <code>torch.manual_seed</code> · <code>nn.Sequential</code> · <code>nn.Conv2d</code> · <code>nn.ReLU</code> · <code>nn.MaxPool2d</code> · <code>nn.AdaptiveAvgPool2d</code> · <code>nn.Flatten</code> · <code>nn.Linear</code></summary>
<p><b><code>torch.manual_seed(seed)</code></b><br>Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</p><pre>&gt;&gt;&gt; _ = torch.manual_seed(0); a = torch.rand(2)
&gt;&gt;&gt; _ = torch.manual_seed(0); b = torch.rand(2)
&gt;&gt;&gt; torch.equal(a, b)
True</pre><p><b><code>nn.Sequential(layer1, layer2, ...)</code></b><br>A model made of layers applied one after the other, the output of each being the input of the next. Saves writing a class when forward() would only chain layers. The layers can be read back by position: model[0].</p><pre>&gt;&gt;&gt; model = nn.Sequential(nn.Flatten(), nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))
&gt;&gt;&gt; model(torch.zeros(5, 2, 2)).shape      # (5, 2, 2) -&gt; (5, 4) -&gt; (5, 3) -&gt; (5, 2)
torch.Size([5, 2])
&gt;&gt;&gt; model[1]
Linear(in_features=4, out_features=3, bias=True)</pre><p>See also: <code>nn.Module</code></p><p><b><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code></b><br>A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</p><pre>&gt;&gt;&gt; conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
&gt;&gt;&gt; conv.weight.shape, conv.bias.shape
(torch.Size([8, 3, 3, 3]), torch.Size([8]))
&gt;&gt;&gt; conv(torch.zeros(2, 3, 28, 28)).shape                 # padding 1 keeps 28 x 28
torch.Size([2, 8, 28, 28])
&gt;&gt;&gt; nn.Conv2d(3, 8, 3, stride=2)(torch.zeros(2, 3, 28, 28)).shape     # (28 - 3) // 2 + 1 = 13
torch.Size([2, 8, 13, 13])</pre><p><i>Watch out:</i> The input must be channels-first, with in_channels channels: a (B, H, W, C) batch fails with &quot;expected input ... to have 3 channels, but got 28 channels instead&quot;; permute it first.</p><p>See also: <code>nn.MaxPool2d</code>, <code>Tensor.permute</code></p><p><b><code>nn.ReLU()</code></b><br>The ReLU activation as a layer (a module): max(x, 0) element by element, no parameters. Used inside nn.Sequential; in a forward() you can write F.relu(x) instead.</p><pre>&gt;&gt;&gt; nn.ReLU()(torch.tensor([-2., 0., 3.]))
tensor([0., 0., 3.])</pre><p>See also: <code>F.relu</code>, <code>nn.Sequential</code></p><p><b><code>nn.MaxPool2d(kernel_size, stride=None)</code></b><br>Max pooling: keeps the largest value of every kernel_size x kernel_size window, separately for each channel; it has no parameters. The stride defaults to kernel_size, so nn.MaxPool2d(2) halves the height and width: (B, C, 28, 28) -&gt; (B, C, 14, 14).</p><pre>&gt;&gt;&gt; x = torch.arange(16.).reshape(1, 1, 4, 4)
&gt;&gt;&gt; nn.MaxPool2d(2)(x)
tensor([[[[ 5.,  7.],
          [13., 15.]]]])
&gt;&gt;&gt; nn.MaxPool2d(2)(torch.zeros(1, 64, 7, 7)).shape       # odd sizes are rounded down
torch.Size([1, 64, 3, 3])</pre><p>See also: <code>nn.Conv2d</code></p><p><b><code>nn.AdaptiveAvgPool2d(output_size)</code></b><br>Average-pools a (N, C, H, W) tensor down to a fixed output size, whatever H and W are: you give the output size, it chooses the windows. With output size 1 it is global average pooling: every channel is averaged over all positions, (N, C, H, W) -&gt; (N, C, 1, 1); .flatten(1) then gives (N, C) for a linear layer.</p><pre>&gt;&gt;&gt; x = torch.arange(8.0).view(1, 2, 2, 2)              # 2 channels of 2 x 2
&gt;&gt;&gt; nn.AdaptiveAvgPool2d(1)(x).flatten(1)               # the mean of each channel
tensor([[1.5000, 5.5000]])
&gt;&gt;&gt; nn.AdaptiveAvgPool2d(1)(torch.zeros(2, 128, 8, 8)).shape
torch.Size([2, 128, 1, 1])
&gt;&gt;&gt; nn.AdaptiveAvgPool2d(1)(torch.zeros(2, 128, 5, 7)).shape    # any input size
torch.Size([2, 128, 1, 1])</pre><p>See also: <code>nn.MaxPool2d</code>, <code>Tensor.flatten</code></p><p><b><code>nn.Flatten(start_dim=1)</code></b><br>A layer that flattens everything except the batch dimension: (B, C, H, W) -&gt; (B, C*H*W). Goes between convolutional layers and an nn.Linear, or in front of a dense network that reads raw pixels.</p><pre>&gt;&gt;&gt; nn.Flatten()(torch.zeros(8, 1, 28, 28)).shape
torch.Size([8, 784])</pre><p>See also: <code>Tensor.flatten</code>, <code>nn.Linear</code></p><p><b><code>nn.Linear(in_features, out_features, bias=True)</code></b><br>A fully connected layer: y = x @ W.T + b, with a learned weight W of shape (out_features, in_features) and a bias b of shape (out_features,). It acts on the last dimension, which must have size in_features, and keeps the others: (B, in) -&gt; (B, out).</p><pre>&gt;&gt;&gt; lin = nn.Linear(3, 2)
&gt;&gt;&gt; lin.weight.shape, lin.bias.shape
(torch.Size([2, 3]), torch.Size([2]))
&gt;&gt;&gt; lin(torch.zeros(8, 3)).shape           # a batch of 8 vectors of length 3
torch.Size([8, 2])</pre><p><i>Watch out:</i> An image batch (B, C, H, W) must first become (B, features), with nn.Flatten or a global average pool; a wrong size fails with &quot;mat1 and mat2 shapes cannot be multiplied&quot;.</p><p>See also: <code>nn.Flatten</code>, <code>Module.parameters</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
torch.manual_seed(0)                                     # the same random starting weights every time
scratch_net = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),          # 3 colour channels in -> 16 maps: 16 x 128 x 128
    nn.ReLU(),
    nn.MaxPool2d(2),                                     # 16 x 64 x 64
    nn.Conv2d(16, 32, kernel_size=3, padding=1),         # 32 x 64 x 64
    nn.ReLU(),
    nn.MaxPool2d(2),                                     # 32 x 32 x 32
    nn.Conv2d(32, 64, kernel_size=3, padding=1),         # 64 x 32 x 32
    nn.ReLU(),
    nn.MaxPool2d(2),                                     # 64 x 16 x 16
    nn.Conv2d(64, 64, kernel_size=3, padding=1),         # 64 x 16 x 16
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),                             # average of each map: 64 x 1 x 1
    nn.Flatten(),                                        # 64 numbers
    nn.Linear(64, number_of_classes),                    # one score per breed
)
history_scratch = P.train(scratch_net, train_loader, val_loader, epochs=10, lr=0.001,
                          name="from scratch: small CNN")

**Reading the curves.** The validation accuracy creeps from 27.0% after the first epoch to its best, 42.0%, in
epoch 9, and jumps up and down on the way (31.0% in epoch 8). That is better than guessing (1 breed in 5 is
20%), but more than half of the photos are wrong. The training accuracy is no better (41.0% at the end): the
network has not even learned the photos it trains on. 300 photos are far too few to learn from nothing what a
pug or a Siamese looks like.

`P.remember(name, model, history)` records this result for the leaderboard at the end of the notebook.

In [ ]:
P.remember("from scratch: small CNN", scratch_net, history_scratch)

## 4 · Step 1: download a pretrained network

A **pretrained** network is one that somebody else has already trained. Its **weights** (all its learned
numbers) come as a file that you download. `torchvision.models` holds dozens of such networks.
`torchvision.models.resnet18(weights=...)` builds ResNet-18 and fills it with the weights named by
`ResNet18_Weights.IMAGENET1K_V1`: the ones learned on ImageNet.

**The first time**, this line downloads the weights file, about 45 MB, into PyTorch's cache folder (on
Windows `C:\Users\<you>\.cache\torch\hub\checkpoints`, on Linux and macOS `~/.cache/torch/hub/checkpoints`).
Every later time it loads the file from there in about a second. Without an internet connection, copy the
file `resnet18-f37072fd.pth` from a computer that has it into that folder.

`print(model.fc)` shows the last layer, the classifier.

<details class="api-ref" data-api="torchvision.models.resnet18,torchvision.models.ResNet18_Weights"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.resnet18</code> · <code>torchvision.models.ResNet18_Weights</code></summary>
<p><b><code>torchvision.models.resnet18(weights=None, num_classes=1000)</code></b><br>Builds a ResNet-18: a stem (conv1 7x7 stride 2, bn1, relu, maxpool), four stages layer1 .. layer4 of two basic residual blocks each (64, 128, 256, 512 channels), global average pooling (avgpool) and a linear head fc (512 -&gt; num_classes); 11.7 M parameters. weights=None gives random weights and downloads nothing; weights=&quot;DEFAULT&quot; downloads the ImageNet weights. Thanks to the global pooling it accepts other input sizes than 224 px. Every torchvision model works this way: resnet50, mobilenet_v3_large, efficientnet_b0, convnext_tiny, ...</p><pre>&gt;&gt;&gt; r18 = torchvision.models.resnet18(weights=None)
&gt;&gt;&gt; [name for name, _ in r18.named_children()]
[&#x27;conv1&#x27;, &#x27;bn1&#x27;, &#x27;relu&#x27;, &#x27;maxpool&#x27;, &#x27;layer1&#x27;, &#x27;layer2&#x27;, &#x27;layer3&#x27;, &#x27;layer4&#x27;, &#x27;avgpool&#x27;, &#x27;fc&#x27;]
&gt;&gt;&gt; r18.fc
Linear(in_features=512, out_features=1000, bias=True)
&gt;&gt;&gt; r18(torch.zeros(2, 3, 64, 64)).shape       # a 64 px input works too
torch.Size([2, 1000])</pre><p>See also: <code>torchvision.models.get_model</code>, <code>torchvision.models.vgg16</code></p><p><b><code>torchvision.models.ResNet18_Weights.IMAGENET1K_V1</code></b><br>Names one set of trained weights for ResNet-18: IMAGENET1K_V1 are the weights learned on ImageNet (1.28 million photos, 1 000 classes); .DEFAULT is the recommended set (here the same one). Passing it to torchvision.models.resnet18(weights=...) downloads the file once (about 45 MB) into PyTorch&#x27;s cache folder and loads it. The object also knows the preparation the weights expect (.transforms()) and the 1 000 class names (.meta[&quot;categories&quot;]); reading those downloads nothing. Every torchvision model has such a class: ResNet50_Weights, MobileNet_V3_Large_Weights, ...</p><pre>&gt;&gt;&gt; w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
&gt;&gt;&gt; len(w.meta[&quot;categories&quot;]), w.meta[&quot;categories&quot;][:3]
(1000, [&#x27;tench&#x27;, &#x27;goldfish&#x27;, &#x27;great white shark&#x27;])
&gt;&gt;&gt; w.transforms().mean, w.transforms().std        # the normalisation the weights expect
([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
&gt;&gt;&gt; torchvision.models.ResNet18_Weights.DEFAULT == w
True</pre><p><i>Watch out:</i> weights=None gives the same network with random weights and downloads nothing: use it to rebuild a network whose trained weights you load yourself with load_state_dict.</p><p>See also: <code>torchvision.models.resnet18</code>, <code>Module.load_state_dict</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
print(model.fc)

The classifier `fc` is a linear layer: 512 numbers in, 1 000 scores out, one per ImageNet class.

`P.show_model(model, input_shape, depth=1)` draws the network one block per box (P1 explains the drawing;
`depth=1` keeps each of the blocks `layer1` to `layer4`, a stack of 4 or 5 convolution layers with BatchNorm
and ReLU, as one stacked box).

In [ ]:
P.show_model(model, (3, 128, 128), depth=1, title="ResNet-18 as downloaded")

**What to look at.** The photo goes through a first convolution (`conv1`, with `bn1`, a **BatchNorm** layer
that keeps the numbers in a good range) and pooling, then the four blocks `layer1` to `layer4`. The picture
shrinks (64 x 32 x 32 after the pooling, 512 x 4 x 4 after `layer4`) while the number of feature maps grows
(64 to 512). `AvgPool` (global average pooling) turns the 512 maps into 512 numbers: the features. Then `fc`.

So the **feature extractor** is everything up to the pooling, and the **classifier** is `fc`. `P.summary`
lists the same blocks with their parameters:

In [ ]:
P.summary(model, (3, 128, 128), depth=1)

The feature extractor holds 11 176 512 parameters (all rows except `fc`), the classifier 512 x 1 000 + 1 000
= 513 000. `layer4` alone has 8 393 728, 72% of the feature extractor: the deepest block has the most
feature maps (512) and so the most filter weights. `P.cost` measures what the network costs on this computer
(P1 explains the report):

In [ ]:
P.cost(model, (3, 128, 128), name="ResNet-18", epoch_images=300)

ResNet-18 has 192 times the parameters of our small network (11 689 512 against 60 837) and needs 11 times
the multiply-adds per photo (593 million against 54 million). Training all of it would take about 5.3 s per
epoch of 300 photos on our computer (yours prints its own time). We will train much less of it.

**What has it learned?** `P.show_filters(model)` draws the 64 filters of the first convolution `conv1` as
small colour pictures (each 7 x 7 pixels). Nobody designed them: they came out of the ImageNet training.

In [ ]:
P.show_filters(model, title="the 64 filters of conv1, learned on ImageNet")

Many filters are grey lines and edges at different angles, a dark band next to a light one (for example
filters 14, 20, 41 and 44); others respond to colours or to one colour next to another (orange against blue
in filters 5, 23 and 31, magenta against green in 6 and 16). Each of them finds its pattern anywhere in a
photo, and the deeper blocks combine them into textures and parts. This is the knowledge we transfer.

**The preparation the weights expect.** The weights were learned on photos with values 0 to 1, each colour
channel normalised with the ImageNet mean (0.485, 0.456, 0.406) and spread (0.229, 0.224, 0.225): that is why
`preparation` uses exactly these numbers. Photos prepared another way give no error, only worse answers,
because every number is off from what the weights are used to. The size is more flexible: ImageNet photos
were 224 x 224 pixels, and thanks to the global average pooling the network accepts our 128 x 128 too
(faster on a laptop, a little less accurate).

## 5 · Step 2: freeze it

Every parameter tensor (a weight tensor or a bias tensor of one layer) has a switch, `requires_grad`. When it
is `True`, training computes how this tensor should change and changes it: the tensor is **trainable**. When
it is `False`, training leaves the tensor as it is: it is **frozen**. Freezing the whole network protects
what it learned on ImageNet. (The new classifier of step 3 will be trainable again.)

`count_trainable(model)` adds up the numbers in all trainable tensors: `model.parameters()` goes through every
parameter tensor of the network, and `param.numel()` is how many numbers one tensor holds.

<details class="api-ref" data-api="Module.parameters,Tensor.requires_grad,Tensor.numel"><summary><b>Functions used here</b> (click to open): <code>Module.parameters</code> · <code>Tensor.requires_grad</code> · <code>Tensor.numel</code></summary>
<p><b><code>model.parameters()</code></b><br>All learnable tensors of a model (the weights and biases of every layer inside it), one after the other. What you hand to an optimiser: torch.optim.SGD(model.parameters(), lr=0.1).</p><pre>&gt;&gt;&gt; model = nn.Linear(3, 2)
&gt;&gt;&gt; [tuple(p.shape) for p in model.parameters()]       # weight (out, in), bias (out,)
[(2, 3), (2,)]
&gt;&gt;&gt; sum(p.numel() for p in model.parameters())         # the number of parameters
8</pre><p><i>Watch out:</i> It returns a generator, which can be read only once: wrap it in list(...) if you need it twice.</p><p>See also: <code>torchkit.count_parameters</code></p><p><b><code>x.requires_grad</code></b><br>An attribute: True if autograd records the operations done with this tensor, so that backward() can later compute its .grad. The weights of nn layers have it, data tensors do not, and a result computed from at least one such tensor has it too. Set it when creating a tensor: torch.tensor(2.0, requires_grad=True).</p><pre>&gt;&gt;&gt; torch.ones(2).requires_grad, nn.Linear(2, 1).weight.requires_grad
(False, True)
&gt;&gt;&gt; w = torch.tensor([1., 2.], requires_grad=True)
&gt;&gt;&gt; (w * 3).requires_grad                  # computed from w, so tracked too
True</pre><p>See also: <code>Tensor.grad</code>, <code>Tensor.backward</code>, <code>torch.no_grad</code></p><p><b><code>x.numel()</code></b><br>The total number of elements of a tensor: the product of its shape. Summed over model.parameters() it gives the number of parameters of a model.</p><pre>&gt;&gt;&gt; torch.zeros(64, 3, 7, 7).numel()           # the weight of a 7x7 conv, 3 -&gt; 64 channels
9408
&gt;&gt;&gt; lin = nn.Linear(512, 37)
&gt;&gt;&gt; sum(p.numel() for p in lin.parameters())   # 512 x 37 weights + 37 biases
18981</pre><p>See also: <code>torchkit.count_parameters</code>, <code>Tensor.shape</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def count_trainable(model):
    """How many numbers (weights and biases) training may change."""
    total = 0
    for param in model.parameters():                 # every weight tensor and bias tensor of the network
        if param.requires_grad:                      # trainable?
            total = total + param.numel()            # add the number of values in this tensor
    return total

**Your line (1 of 3).** Inside the loop, replace `____` by `param.requires_grad = False`. Then run the cell,
and the check after it.

In [ ]:
print(f"trainable parameters before freezing: {count_trainable(model):,}")
for param in model.parameters():                 # every parameter tensor of the network, one after the other
    param.requires_grad = False                  # frozen: training will not change this tensor
print(f"trainable parameters after freezing:  {count_trainable(model):,}")

In [ ]:
T.check_frozen(model)

All 11 689 512 parameters are now frozen: trained as it is, the network would not change at all.

## 6 · Step 3: a new classifier

Replace the ImageNet classifier by a new linear layer for our breeds: 512 features in (what the feature
extractor gives), one score per breed out. Assigning to `model.fc` swaps the layer; the rest of the network
stays as it is. A new layer starts with random weights and is trainable (`requires_grad` is `True`), so
after this step exactly its parameters are trainable: 512 x 5 weights + 5 biases = 2 565.

**Your line (2 of 3).** Replace `____` by `nn.Linear(512, number_of_classes)`.

In [ ]:
torch.manual_seed(0)                             # the same random starting weights for the new layer every time
model.fc = nn.Linear(512, number_of_classes)     # 512 features in -> one score per breed out

In [ ]:
T.check_new_head(model, number_of_classes)

In [ ]:
P.show_model(model, (3, 128, 128), depth=1, title="ResNet-18 with a new classifier for 5 breeds")

The same network, except the last box: `Linear 512 -> 5`, 2 565 parameters. Of the 11 179 077 parameters,
only those 2 565 will learn.

**Before training**, the new classifier has random weights. `P.evaluate(model, loader)` measures the loss
and the accuracy on all photos of a loader without changing anything. With 5 breeds, guessing gets 1 photo in
5 right: 20%, the **chance level**.

In [ ]:
loss, accuracy = P.evaluate(model, val_loader)
print(f"before training: validation accuracy {accuracy:.1f}%")

12.5%: no better than guessing (here even a little worse, by chance). The features are good, but the random
classifier does not know yet which features belong to which breed. That is what it learns next.

## 7 · Step 4: train only the new classifier

The **optimiser** is the method that changes the trainable numbers after every batch; here **Adam**, the one
`P.train` uses (P1). It changes exactly the tensors you hand to it. `trainable_parameters(model)` collects the
trainable tensors (only `fc.weight` and `fc.bias` now), and `T.check_optimizer` confirms that the optimiser got
them and nothing else. `P.train` normally builds the optimiser itself; with `optimizer=` it uses ours.

<details class="api-ref" data-api="torch.optim.Adam"><summary><b>Functions used here</b> (click to open): <code>torch.optim.Adam</code></summary>
<p><b><code>torch.optim.Adam(params, lr=0.001)</code></b><br>The Adam optimizer. Like SGD it follows the gradient, but it divides each parameter&#x27;s step by a running estimate of that gradient&#x27;s size, so a step moves about lr whatever the scale of the gradient. D2-01 uses it to move four box coordinates with the same step length for every loss; D2-04 to train free queries.</p><pre>&gt;&gt;&gt; p = torch.tensor([5.0], requires_grad=True)
&gt;&gt;&gt; opt = torch.optim.Adam([p], lr=1.0)
&gt;&gt;&gt; loss = (p ** 2).sum()                  # gradient 2 * 5 = 10
&gt;&gt;&gt; opt.zero_grad(); loss.backward(); opt.step()
&gt;&gt;&gt; p.detach()                             # moved by lr = 1 (SGD would move by lr * 10)
tensor([4.])</pre><p><i>Watch out:</i> params must be an iterable: torch.optim.Adam([p]) for single tensors, model.parameters() for a model. Adam(p) with a bare tensor raises a TypeError.</p><p>See also: <code>torch.optim.AdamW</code>, <code>torch.optim.SGD</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def trainable_parameters(model):
    """The parameter tensors that training may change: what the optimiser gets."""
    found = []
    for param in model.parameters():
        if param.requires_grad:
            found.append(param)
    return found


optimizer = torch.optim.Adam(trainable_parameters(model), lr=0.001)
T.check_optimizer(optimizer, model)

Now train for 5 epochs with learning rate 0.001. Every photo still goes through the whole network, but the
training step only has to work out the changes for the 2 565 numbers of the classifier, so it is fast.

**Expected time: about 17 seconds** (up to a minute on a slow laptop).

In [ ]:
history_head = P.train(model, train_loader, val_loader, epochs=5, lr=0.001, optimizer=optimizer,
                       name="ResNet-18: new classifier only")
P.remember("ResNet-18: new classifier only", model, history_head)

**Reading the curves.** After one epoch the validation accuracy is already 77.5%; after four it reaches
92.5%, its best, and stays there. The network trained from scratch never passed 42.0% in 10 epochs. Same
photos, same number of training steps per epoch: the difference is everything ResNet-18 learned on ImageNet.
(The training accuracy starts lower than the validation accuracy because it is averaged over the epoch while
the classifier is still learning; the validation accuracy is measured at the end of the epoch.)

`P.show_predictions(model, loader, classes)` shows the network's answer for the first 16 validation photos:
green when right, with how sure it is (in percent), red and marked WRONG when wrong.

In [ ]:
P.show_predictions(model, val_loader, breeds)

15 of the 16 photos are right. The one mistake is a pug lying on a striped blanket, called a Bombay (a black
cat). The percentages vary: the network is sure about clear photos (the beagle by the sea, 100%) and unsure
about unusual ones (the samoyed puppy, 36%, although right).

`P.confusion(model, loader, classes)` counts, for every true breed (row), which breed the network answered
(column): the grey diagonal holds the correct answers, orange cells the mistakes. It also returns the counts
as a table of numbers; we keep them in `counts`.

In [ ]:
counts = P.confusion(model, val_loader, breeds, title="rows: true breed, columns: answer")

15 of the 200 validation photos are wrong (7.5%), at most 2 in any one cell. Pugs are the hardest: 34 of 40
right, the others called beagle, samoyed, Bombay or Siamese.

**Side by side with the network trained from scratch.** `P.compare` puts their size, speed and best validation
accuracy in one table and chart, and `P.plot_history` draws their curves on top of each other (solid lines:
validation, dashed: training).

In [ ]:
P.compare({"from scratch": scratch_net, "ResNet-18, new classifier": model}, (3, 128, 128),
          histories={"from scratch": history_scratch, "ResNet-18, new classifier": history_head},
          epoch_images=300)
P.plot_history({"from scratch": history_scratch, "ResNet-18, new classifier": history_head})

**What the numbers say.** The pretrained network is much bigger: 184 times the parameters (11 179 077
against 60 837) and 11 times the multiply-adds, so it takes several times as long per photo (on our
computer, with nothing else running: 5.51 ms against 1.29 ms). Its training step is much cheaper than with
the whole network trainable (there: 210 ms for 32 photos, against the 570 ms that `P.cost` measured before
freezing): for frozen layers there is nothing to work out backwards. For that price the best validation
accuracy goes from 42.0% to 92.5%. Times depend on your computer and on what else runs on it; parameters,
multiply-adds and accuracies do not.

## 8 · Step 5: unfreeze the last block and train a little more

The feature extractor was made for ImageNet's 1 000 classes. Its last block, `layer4`, holds the most
specific patterns (object parts); letting it adapt to our breeds can make the features fit better. This is
called **fine-tuning**. We unfreeze only `layer4` and keep the earlier blocks frozen: edges and textures are
useful as they are.

Then train again, with a **smaller learning rate** (0.0001 instead of 0.001). `layer4` already holds good
weights; big changes, driven by only 300 photos, would destroy what it learned on ImageNet instead of
adjusting it.

**Your line (3 of 3).** Inside the loop, replace `____` by `param.requires_grad = True`.

In [ ]:
for param in model.layer4.parameters():          # only the parameter tensors of the last block, layer4
    param.requires_grad = True                   # trainable again
print(f"trainable parameters now: {count_trainable(model):,}")

In [ ]:
T.check_unfrozen(model, ["layer4"])

Now 8 396 293 numbers are trainable (8 393 728 in `layer4` + 2 565 in `fc`). An optimiser only changes the
tensors it was given, and the one of step 4 knows only `fc`: build a **new** optimiser after unfreezing.

**Expected time: about 20 seconds** (up to a minute and a half on a slow laptop): the training step now also
works out the changes for `layer4`.

In [ ]:
optimizer = torch.optim.Adam(trainable_parameters(model), lr=0.0001)     # a new optimiser, 10 times smaller steps
T.check_optimizer(optimizer, model)
history_finetune = P.train(model, train_loader, val_loader, epochs=5, lr=0.0001, optimizer=optimizer,
                           name="ResNet-18: + layer4 unfrozen")
P.remember("ResNet-18: + layer4 unfrozen", model, history_finetune)

**Reading the curves.** The run starts from the trained classifier, so its first epoch is already at 96.5%;
the second reaches 98.0%, the best, and the last three stay at 98.0% and 97.5%. The training accuracy is
100.0% from epoch 2 on and the training loss falls to 0.002, while the validation loss stays near 0.105: with
8.4 million trainable numbers and only 300 photos, the network now knows its training photos by heart. More
epochs would add little.

The two runs of the ResNet side by side, and the mistakes that are left:

In [ ]:
P.plot_history({"new classifier only": history_head, "+ layer4 unfrozen": history_finetune})
counts = P.confusion(model, val_loader, breeds, title="after unfreezing layer4")

**Is unfreezing worth it?** Here, yes: 5 mistakes out of 200 instead of 15 (97.5% at the end, 98.0% at best,
against 92.5%), for 20 more seconds of training. Read such numbers with care, though: with 200 validation
photos, one photo is 0.5%, so a difference of 2 or 3 photos can be luck. The experiments of section 10 show
the limits: unfreezing `layer3` as well adds time but no accuracy here, and too big a learning rate makes the
network worse than before unfreezing. Unfreezing pays off most when
you have more photos (thousands rather than hundreds) and when your photos look unlike ImageNet's (X-rays,
satellite images, microscope slides): then the ImageNet features fit less well and need to adapt. With few
photos of everyday things, training only the classifier is often enough, and it is faster and safer.

## 9 · Use it

### Name the breed in a photo file

`predict_photo` is what you would use in an application: read one photo file, prepare it exactly like the
training photos, and let the network answer. Three new things in it:

* `x.unsqueeze(0)` turns one photo (3 x 128 x 128) into a batch of one photo (1 x 3 x 128 x 128): networks
  always take batches;
* `model.eval()` switches the network to **evaluation mode**: the BatchNorm layers then use the averages they
  stored during training instead of those of the current batch, so the answer for a photo does not depend on
  the other photos (`P.train` and `P.evaluate` do this for you); `torch.no_grad()` tells PyTorch not to
  prepare for training, which saves time and memory;
* `scores.softmax(dim=1)` turns the 5 scores into 5 **probabilities**: numbers between 0 and 1 that add up to
  1. The highest one is the network's **confidence** in its answer.

<details class="api-ref" data-api="Tensor.unsqueeze,Module.eval,torch.no_grad,Tensor.softmax,Tensor.argmax"><summary><b>Functions used here</b> (click to open): <code>Tensor.unsqueeze</code> · <code>Module.eval</code> · <code>torch.no_grad</code> · <code>Tensor.softmax</code> · <code>Tensor.argmax</code></summary>
<p><b><code>x.unsqueeze(dim)</code></b><br>Inserts a new dimension of size 1 at position dim; the data is untouched. Its main use: a model expects a batch (N, C, H, W), you have one image (C, H, W), so add the batch dimension.</p><pre>&gt;&gt;&gt; img = torch.zeros(3, 32, 48)         # C, H, W
&gt;&gt;&gt; img.unsqueeze(0).shape               # 1, C, H, W: a batch of one
torch.Size([1, 3, 32, 48])
&gt;&gt;&gt; v = torch.tensor([1., 2., 3.])
&gt;&gt;&gt; v.unsqueeze(1).shape                 # a column: (3,) -&gt; (3, 1)
torch.Size([3, 1])
&gt;&gt;&gt; v[None].shape                        # indexing with None does the same as unsqueeze(0)
torch.Size([1, 3])</pre><p><i>Watch out:</i> unsqueeze returns a new view; x itself keeps its shape unless you write x = x.unsqueeze(0).</p><p>See also: <code>Tensor.squeeze</code>, <code>Tensor.view</code></p><p><b><code>model.eval()</code></b><br>Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</p><pre>&gt;&gt;&gt; drop = nn.Dropout(p=0.5)
&gt;&gt;&gt; x = torch.ones(1, 6)
&gt;&gt;&gt; _ = torch.manual_seed(0)
&gt;&gt;&gt; drop(x)                         # train mode (the default): random zeros, the rest x 2
tensor([[0., 0., 2., 0., 0., 0.]])
&gt;&gt;&gt; _ = drop.eval()
&gt;&gt;&gt; drop(x)                         # eval mode: unchanged
tensor([[1., 1., 1., 1., 1., 1.]])</pre><p><i>Watch out:</i> eval() does not stop gradient tracking: wrap inference in torch.no_grad() as well. The mode stays until you change it: call model.train() before training again.</p><p>See also: <code>Module.train</code>, <code>torch.no_grad</code></p><p><b><code>with torch.no_grad(): ...</code></b><br>A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</p><pre>&gt;&gt;&gt; w = torch.ones(2, requires_grad=True)
&gt;&gt;&gt; (w * 2).requires_grad
True
&gt;&gt;&gt; with torch.no_grad():
...     y = w * 2
&gt;&gt;&gt; y.requires_grad
False</pre><p><i>Watch out:</i> It does not switch layers such as dropout and batch norm to evaluation mode: that is model.eval(), and evaluation code uses both.</p><p>See also: <code>Tensor.requires_grad</code>, <code>Module.eval</code></p><p><b><code>logits.softmax(dim)</code></b><br>Turns scores (logits) into probabilities along dimension dim: exp(x) / sum(exp(x)). Every value ends up between 0 and 1 and they sum to 1 along dim; the order of the scores is kept, so the argmax does not change.</p><pre>&gt;&gt;&gt; logits = torch.tensor([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
&gt;&gt;&gt; logits.softmax(1)                          # one probability vector per row
tensor([[0.0900, 0.2447, 0.6652],
        [0.3333, 0.3333, 0.3333]])
&gt;&gt;&gt; logits.softmax(1).sum(1)
tensor([1., 1.])</pre><p><i>Watch out:</i> Always give dim (1 for a batch of logits (N, C)). For training, pass the raw logits to F.cross_entropy, not the softmax output.</p><p>See also: <code>F.log_softmax</code>, <code>torch.topk</code></p><p><b><code>x.argmax(dim)</code></b><br>The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</p><pre>&gt;&gt;&gt; logits = torch.tensor([[0.1, 2.0, -1.0], [3.0, 0.0, 0.5]])     # B=2 samples, 3 classes
&gt;&gt;&gt; logits.argmax(dim=1)                   # predicted class per sample
tensor([1, 0])
&gt;&gt;&gt; logits.argmax()                        # no dim: index into the 6 flattened values
tensor(3)</pre><p><i>Watch out:</i> Softmax does not change which value is largest, so predictions are read off the logits directly.</p><p>See also: <code>Tensor.max</code>, <code>torchkit.accuracy</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def predict_photo(model, path, breeds, image_size):
    """The breed the model sees in one photo file, and how sure it is (0 to 1)."""
    photo = Image.open(path).convert("RGB")
    x = preparation(image_size)(photo)              # the same preparation as the training photos: 3 x H x W
    batch = x.unsqueeze(0)                          # a batch of one photo: 1 x 3 x H x W
    model.eval()                                    # evaluation mode
    with torch.no_grad():                           # only answering, no training
        scores = model(batch)                       # 1 x 5 scores
    probabilities = scores.softmax(dim=1)[0]        # 5 numbers between 0 and 1 that add up to 1
    best = int(probabilities.argmax())              # the position of the highest probability
    return breeds[best], float(probabilities[best])


photo_folder = torchkit.data_root() / "oxford-iiit-pet" / "images"
for breed in breeds:
    files = sorted(photo_folder.glob(breed + "_*.jpg"))
    path = files[-1]                                # the last photo of this breed: one of the validation photos
    predicted, confidence = predict_photo(model, path, breeds, image_size)
    print(f"{path.name:<16} the network says: {predicted:<8} ({100 * confidence:.1f}% sure)")

All five right, four of them at 98.8% or more and the pug at 96.8%. To use the network on your own photo,
pass its file name as `path`.

### Save it and load it again

`model.state_dict()` holds all the numbers of the network, by name; `torch.save` writes them to a file. To use
the network later (in another notebook or program), build the same network with `weights=None` (random
weights, nothing downloaded), give it the same new classifier, and load the saved numbers into it with
`load_state_dict`. The file goes into the folder of this notebook.

<details class="api-ref" data-api="Module.state_dict,torch.save,torch.load,Module.load_state_dict"><summary><b>Functions used here</b> (click to open): <code>Module.state_dict</code> · <code>torch.save</code> · <code>torch.load</code> · <code>Module.load_state_dict</code></summary>
<p><b><code>model.state_dict()</code></b><br>An ordered dict from names to tensors holding everything that defines the model&#x27;s state: all parameters (weights, biases) and buffers (BatchNorm&#x27;s running_mean and running_var). Names follow the module structure, e.g. &quot;layer1.0.conv1.weight&quot;. Optimizers and schedulers have a state_dict() too (momentum, Adam&#x27;s moment estimates, the step counter): to resume training, save all three.</p><pre>&gt;&gt;&gt; net = nn.Sequential(nn.Linear(3, 2), nn.BatchNorm1d(2))
&gt;&gt;&gt; {k: tuple(v.shape) for k, v in net.state_dict().items()}
{&#x27;0.weight&#x27;: (2, 3), &#x27;0.bias&#x27;: (2,), &#x27;1.weight&#x27;: (2,), &#x27;1.bias&#x27;: (2,), &#x27;1.running_mean&#x27;: (2,), &#x27;1.running_var&#x27;: (2,), &#x27;1.num_batches_tracked&#x27;: ()}</pre><p><i>Watch out:</i> The tensors share memory with the model, so further training changes them: to keep a snapshot in memory use copy.deepcopy(model.state_dict()); torch.save writes the current values to disk.</p><p>See also: <code>Module.load_state_dict</code>, <code>torch.save</code>, <code>Module.named_parameters</code></p><p><b><code>torch.save(obj, path)</code></b><br>Writes a Python object that contains tensors (a tensor, a state_dict, a dict of several state_dicts and numbers, ...) to a file path or an open file object, in Python&#x27;s pickle format. The usual checkpoint is a dict: {&quot;model&quot;: model.state_dict(), &quot;optimizer&quot;: opt.state_dict(), &quot;epoch&quot;: epoch}.</p><pre>&gt;&gt;&gt; import io
&gt;&gt;&gt; buf = io.BytesIO()               # an in-memory file; in practice a path such as &quot;ckpt.pt&quot;
&gt;&gt;&gt; torch.save({&quot;w&quot;: torch.tensor([1.0, 2.0]), &quot;epoch&quot;: 3}, buf)
&gt;&gt;&gt; _ = buf.seek(0)                  # rewind before reading
&gt;&gt;&gt; torch.load(buf)
{&#x27;w&#x27;: tensor([1., 2.]), &#x27;epoch&#x27;: 3}</pre><p><i>Watch out:</i> Save state_dict()s rather than whole model objects: a pickled model only loads again where its class definition can be imported, unchanged, under the same name.</p><p>See also: <code>torch.load</code>, <code>Module.state_dict</code>, <code>torchkit.save_checkpoint</code></p><p><b><code>torch.load(path, map_location=None, weights_only=True)</code></b><br>Reads back what torch.save wrote. By default (weights_only=True) it only accepts tensors, numbers, strings and lists / tuples / dicts of them: a safety measure, because a pickle file can run code. weights_only=False loads any object, for files you trust. map_location=&quot;cpu&quot; puts tensors that were saved on a GPU onto the CPU.</p><pre>&gt;&gt;&gt; import io
&gt;&gt;&gt; buf = io.BytesIO()
&gt;&gt;&gt; torch.save({&quot;model&quot;: nn.Linear(2, 1).state_dict(), &quot;epoch&quot;: 7}, buf)
&gt;&gt;&gt; _ = buf.seek(0)
&gt;&gt;&gt; ck = torch.load(buf, map_location=&quot;cpu&quot;)
&gt;&gt;&gt; sorted(ck), ck[&quot;epoch&quot;], list(ck[&quot;model&quot;])
([&#x27;epoch&#x27;, &#x27;model&#x27;], 7, [&#x27;weight&#x27;, &#x27;bias&#x27;])</pre><p><i>Watch out:</i> A file that also holds other objects (a NumPy array, an instance of your own class) fails with &quot;Weights only load failed&quot; under the default; pass weights_only=False only for files you made yourself.</p><p>See also: <code>torch.save</code>, <code>Module.load_state_dict</code>, <code>torchkit.load_checkpoint</code></p><p><b><code>model.load_state_dict(state_dict, strict=True)</code></b><br>Copies the tensors of a state_dict into an existing model&#x27;s parameters and buffers, matched by name; names and shapes must agree. Returns a report of missing and unexpected names. Optimizers and schedulers have load_state_dict too.</p><pre>&gt;&gt;&gt; a, b = nn.Linear(3, 2), nn.Linear(3, 2)    # two different random initialisations
&gt;&gt;&gt; torch.equal(a.weight, b.weight)
False
&gt;&gt;&gt; b.load_state_dict(a.state_dict())
&lt;All keys matched successfully&gt;
&gt;&gt;&gt; torch.equal(a.weight, b.weight)
True</pre><p><i>Watch out:</i> strict=False skips missing or extra names (handy after replacing a head), but a shape mismatch is still an error. It fills a model you have already built; it does not build one.</p><p>See also: <code>Module.state_dict</code>, <code>torch.load</code>, <code>torchkit.load_checkpoint</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
torch.save(model.state_dict(), "pets_resnet18.pt")                 # save: all the numbers of the adapted network
size_mb = pathlib.Path("pets_resnet18.pt").stat().st_size / 1_000_000
print(f"saved pets_resnet18.pt: {size_mb:.1f} MB")

reloaded = torchvision.models.resnet18(weights=None)               # load: the same network, random weights,
reloaded.fc = nn.Linear(512, number_of_classes)                    # with the same new classifier,
reloaded.load_state_dict(torch.load("pets_resnet18.pt"))           # then the saved numbers put in

predicted, confidence = predict_photo(reloaded, path, breeds, image_size)
print(f"the reloaded network on {path.name}: {predicted} ({100 * confidence:.1f}% sure)")

The file is 44.8 MB: about 4 bytes for each of the 11.2 million numbers. The reloaded network gives the same
answer as the one we trained (beagle, 99.9%): the file holds everything it learned.

## 10 · Try it: the whole recipe with your settings

`run_recipe(...)` repeats the data loading and steps 1 to 5 with the settings you give it, in plain code (the
same lines as above, without the checks). Two details:

* `getattr(model, "layer4")` is the same as `model.layer4`: it lets the block names come from the list
  `unfreeze`;
* `model.fc.in_features` is the number of features the old classifier received: 512 for ResNet-18, 2 048 for
  ResNet-50, so the new classifier fits either network.

It records each training in the leaderboard (`P.remember`) and ends by printing the validation accuracy of the
network you keep.

<details class="api-ref" data-api="torchvision.models.resnet50"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.resnet50</code></summary>
<p><b><code>torchvision.models.resnet50(weights=None)</code></b><br>Builds a ResNet-50, the 50-layer ResNet made of the bottleneck blocks of D1-03 (25.6 M parameters, 2048 features before the classifier). weights=&quot;IMAGENET1K_V2&quot; downloads the ImageNet-trained weights (about 100 MB, once); weights=None gives random ones. Input (N, 3, H, W) normalised images, output (N, 1000) ImageNet logits.</p><pre>&gt;&gt;&gt; net = torchvision.models.resnet50(weights=None).eval()      # random weights, no download
&gt;&gt;&gt; net(torch.zeros(1, 3, 224, 224)).shape
torch.Size([1, 1000])
&gt;&gt;&gt; [name for name, _ in net.named_children()][-3:]             # what comes after layer4
[&#x27;layer4&#x27;, &#x27;avgpool&#x27;, &#x27;fc&#x27;]
&gt;&gt;&gt; net.fc
Linear(in_features=2048, out_features=1000, bias=True)</pre><p>See also: <code>torchvision.models.resnet18</code>, <code>Module.children</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def run_recipe(breeds, photos_per_class, image_size, backbone, epochs_head, learning_rate_head,
               unfreeze, epochs_finetune, learning_rate_finetune):
    """Load the photos, then steps 1 to 5 of the recipe. Returns the adapted network."""
    train_loader, val_loader = load_pets(breeds, photos_per_class, 40, image_size)
    number_of_classes = len(breeds)
    name = f"{backbone} {photos_per_class}/breed {image_size}px"

    # step 1: a pretrained network
    if backbone == "resnet50":
        model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    else:
        model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

    # step 2: freeze every parameter
    for param in model.parameters():
        param.requires_grad = False

    # step 3: a new classifier (trainable)
    torch.manual_seed(0)
    model.fc = nn.Linear(model.fc.in_features, number_of_classes)

    # step 4: train only the new classifier
    if epochs_head > 0:
        optimizer = torch.optim.Adam(trainable_parameters(model), lr=learning_rate_head)
        history = P.train(model, train_loader, val_loader, epochs=epochs_head, lr=learning_rate_head,
                          optimizer=optimizer, name=name + ": classifier only")
        P.remember(name + ": classifier only", model, history)

    # step 5: unfreeze the chosen blocks and train on with a smaller learning rate
    if len(unfreeze) > 0 and epochs_finetune > 0:
        for block_name in unfreeze:
            block = getattr(model, block_name)          # e.g. model.layer4
            for param in block.parameters():
                param.requires_grad = True
        optimizer = torch.optim.Adam(trainable_parameters(model), lr=learning_rate_finetune)
        history = P.train(model, train_loader, val_loader, epochs=epochs_finetune, lr=learning_rate_finetune,
                          optimizer=optimizer, name=name + ": + " + str(unfreeze))
        P.remember(name + ": + " + str(unfreeze), model, history)

    loss, accuracy = P.evaluate(model, val_loader)
    print(f"the network you keep: validation accuracy {accuracy:.1f}% "
          f"({count_trainable(model):,} parameters were trainable in the last step)")
    return model

**Try it.** Change the settings, then run this cell and the next one. The breeds are the starts of the file
names: cats `Abyssinian`, `Bengal`, `Birman`, `Bombay`, `British_Shorthair`, `Egyptian_Mau`, `Maine_Coon`,
`Persian`, `Ragdoll`, `Russian_Blue`, `Siamese`, `Sphynx`; dogs `american_bulldog`,
`american_pit_bull_terrier`, `basset_hound`, `beagle`, `boxer`, `chihuahua`, `english_cocker_spaniel`,
`english_setter`, `german_shorthaired`, `great_pyrenees`, `havanese`, `japanese_chin`, `keeshond`,
`leonberger`, `miniature_pinscher`, `newfoundland`, `pomeranian`, `pug`, `saint_bernard`, `samoyed`,
`scottish_terrier`, `shiba_inu`, `staffordshire_bull_terrier`, `wheaten_terrier`, `yorkshire_terrier`.

In [ ]:
# ---------------- Try it: the data ----------------
breeds = ["Bombay", "Siamese", "pug", "samoyed", "beagle"]   # 2 to 37 breeds (names above)
photos_per_class = 60            # training photos per breed (at most 150)
image_size = 128                 # height = width in pixels, e.g. 96, 128, 160, 224
# ---------------- Try it: the recipe ----------------
backbone = "resnet18"            # or "resnet50": bigger and slower (about 100 MB to download the first time)
epochs_head = 5                  # step 4: epochs that train only the new classifier (0 = no training)
learning_rate_head = 0.001
unfreeze = ["layer4"]            # step 5: blocks to unfreeze: [] (none), ["layer4"] or ["layer3", "layer4"]
epochs_finetune = 5              # step 5: epochs after unfreezing (0 = no step 5)
learning_rate_finetune = 0.0001  # step 5: usually 10 times smaller than learning_rate_head

**Expected time: about 40 seconds** with the settings above (up to 2 minutes on a slow laptop; more photos,
bigger images, more unfrozen blocks and ResNet-50 take longer).

In [ ]:
adapted = run_recipe(breeds, photos_per_class, image_size, backbone, epochs_head, learning_rate_head,
                     unfreeze, epochs_finetune, learning_rate_finetune)

**The leaderboard.** Run this cell after every experiment: every version so far, best validation accuracy
first. With the settings above, the two rows of `run_recipe` repeat steps 4 and 5 exactly.

In [ ]:
P.leaderboard()

### Guided experiments

Do them in any order. For each one: change the setting in the **Try it** cell, **predict** what will happen,
run the two cells, and compare. Then open the fold-out to read what we saw (the other settings as above; your
numbers can differ in the last digit). Put the setting back afterwards.

**1. Fewer photos.** `photos_per_class = 20` (100 training photos in total).
<details><summary>What we saw</summary>

The new classifier alone reaches 88.0% at its best (epoch 4; 84.0% after epoch 5), against 92.5% with 60
photos per breed; unfreezing `layer4` then takes it to 95.5%. Even with 20 photos per breed the pretrained
network is far ahead of the network trained from scratch on 60 per breed (42.0%). This is where transfer
learning shines: when photos are few.
</details>

**2. How much to unfreeze.** `unfreeze = []` (stop after step 4), then `unfreeze = ["layer3", "layer4"]`.
<details><summary>What we saw</summary>

With `[]` the recipe ends after the new classifier: 92.5%, as in step 4. With `["layer3", "layer4"]`,
10 496 005 numbers are trainable (2.1 million more than with `layer4` alone) and each epoch takes longer
(about 5 s instead of about 4 s here). The best validation accuracy is 98.0%, the same as with `layer4` alone
(it ends at 98.0% instead of 97.5%: one photo). On this small set, unfreezing more blocks costs time and
brings no clear gain.
</details>

**3. Fine-tuning with too big a learning rate.** `learning_rate_finetune = 0.01` (100 times the default, 10
times the learning rate of the classifier).
<details><summary>What we saw</summary>

The classifier reaches 92.5% as before. Then the first fine-tuning epoch throws the network back to 68.0%,
with a validation loss of 12.129 (it was 0.281): the big steps wreck what `layer4` learned on ImageNet. It
partly recovers (85.5%, 82.5%, 91.0%), but jumps around and ends at 87.0%, worse than before unfreezing. With
the default 0.0001 the same step took it to 98.0%. That is why fine-tuning uses a small learning rate.
</details>

**4. Image size.** `image_size = 96`, then `image_size = 160`.
<details><summary>What we saw</summary>

At 96 x 96 pixels: 82.5% with the new classifier alone, 90.0% after unfreezing `layer4`: fur and faces lose
their detail. At 160 x 160: 98.0% with the classifier alone and 99.5% after unfreezing (1 photo wrong out of
200). But a 160-pixel photo costs 925 million multiply-adds instead of 592 million, and the epochs took 2 to 3
times as long here (7.1 s and 11.7 s per epoch). Of all the settings, the photo size helps most, closer to
the 224 pixels the network was trained on; the price is time.
</details>

**5. No training at all.** `epochs_head = 0` and `epochs_finetune = 0`.
<details><summary>What we saw</summary>

The pretrained network with its random new classifier: 12.5% on the validation photos, the same as before
step 4, no better than guessing (20%). The feature extractor alone does not know our breeds: the classifier
has to learn which features mean which breed. That takes seconds, but it cannot be skipped.
</details>

**6. Breeds that look alike.** `breeds = ["american_pit_bull_terrier", "staffordshire_bull_terrier",
"american_bulldog", "boxer", "miniature_pinscher"]`.
<details><summary>What we saw</summary>

Five short-haired, muscular dog breeds that people confuse too. The new classifier alone reaches only 66.0%;
unfreezing `layer4` adds 7 points (73.0% at its best, epoch 1, and 71.5% at the end), while the training
accuracy reaches 100%: with 60 photos per breed the network learns its training photos by heart rather than
the fine differences between the breeds. Breeds this close need many more photos per breed, and larger
photos; the recipe stays the same.
</details>

## 11 · What to remember

* **Transfer learning** reuses a network trained on a huge collection (ImageNet: 1.28 million photos) for your
  own classes. Its feature extractor already knows edges, textures and object parts.
* The recipe: **freeze, replace the head, train the head, then maybe unfreeze the last block with a small
  learning rate.** In code: `param.requires_grad = False` in a loop, `model.fc = nn.Linear(512, classes)`,
  train, `param.requires_grad = True` for `model.layer4`, a new optimiser, train again.
* With 300 photos, the network trained from scratch reached 42.0% at best; the pretrained one with a new
  classifier 92.5% after 17 seconds of training, by training only 2 565 of its 11 million numbers, and 98.0%
  after unfreezing `layer4`.
* Prepare your photos the way the weights expect (ImageNet mean and std), and build a new optimiser after
  changing what is frozen: an optimiser only changes the tensors it was given.
* To use the result: `model.eval()`, `torch.no_grad()`, softmax for the confidence; `torch.save(model.state_dict(),
  path)` to keep it, and `load_state_dict` into the same network to load it.

### Words from this notebook

| word | meaning |
|---|---|
| accuracy | percentage of photos whose highest score is the right class |
| batch | a small group of photos processed together (here 32) |
| BatchNorm | a layer that keeps the numbers in a good range; it behaves differently in training and evaluation mode |
| chance level | the accuracy of guessing: 1 / number of classes (20% for 5 breeds) |
| class | one of the possible answers (here a breed) |
| classifier (head) | the last layer: features in, one score per class out |
| confidence | the probability the network gives its answer |
| epoch | one pass over all training photos |
| evaluation mode | `model.eval()`: BatchNorm uses its stored averages, so one photo's answer does not depend on the others |
| feature extractor | all layers before the classifier: photo in, features out |
| feature map | the output of one filter of a convolution: bright where its pattern appears |
| features | the numbers that describe a photo (512 for ResNet-18) |
| fine-tuning | training (part of) a pretrained feature extractor a little more, with a small learning rate |
| from scratch | starting from random weights |
| frozen | a parameter that training does not change (`requires_grad = False`) |
| global average pooling | averaging each feature map over the whole picture into one number |
| ImageNet | 1.28 million photos in 1 000 classes; most pretrained image networks learned from it |
| label | the correct class of a photo, as a number |
| learning rate | how big each change of the weights is |
| loss | how wrong the scores are (lower is better) |
| multiply-adds | the multiplications and additions a network needs for one photo: a measure of its work |
| normalise | subtract a mean and divide by a spread, per colour channel |
| optimiser | the method that changes the trainable numbers after each batch (here Adam) |
| parameters | the numbers a network learns: the weights and biases of its layers, held in tensors |
| pretrained | already trained by someone else; its weights come as a file |
| probability | a number between 0 and 1; the 5 probabilities of a photo add up to 1 (softmax) |
| random seed | the starting point of the random numbers, so results repeat |
| score | one output number per class; the highest is the answer |
| tensor | PyTorch's array of numbers |
| trainable | a parameter that training changes (`requires_grad = True`) |
| transfer learning | reusing a network trained on one task for another |
| unfreeze | make frozen parameters trainable again |
| training / validation set | photos the network learns from / photos kept apart to measure it |
| weights | the learned numbers of a network (its parameters) |